# 04. 5개 Tool 단위 테스트

에이전트가 호출할 5개 tool을 LLM 없이 직접 호출하여 입출력 스키마/동작을 검증.

| Tool | 역할 |
|------|------|
| query_sensor | 시계열 조회 + 통계 요약 |
| detect_anomaly | 이상 점수 + 의심 센서 식별 |
| search_manual | 매뉴얼/SOP RAG |
| search_history | 과거 고장 이력 검색 |
| draft_workorder | 작업지시서 초안 생성 |

In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_metropt3, SensorDB
from src.models.anomaly import AEAnomalyModel
from src.rag.retriever import load_retriever
from src.agent.tools import (
    configure_tools, ToolDeps,
    query_sensor, detect_anomaly, search_manual, search_history, draft_workorder,
)

ARTIFACTS = PROJECT_ROOT / 'models_artifacts'
print('imports OK')

imports OK


## 1. 의존성 준비

- 데이터 로드 : SensorDB
- 학습된 ConvAE 로드 + threshold (02번 노트북 산출물)
- 매뉴얼/이력 retriever 로드 (03번 노트북 산출물)

In [2]:
df = load_metropt3(data_dir=PROJECT_ROOT / 'data' / 'metropt3', downsample='1min')
db = SensorDB(df=df)

ae = AEAnomalyModel.load(ARTIFACTS / 'convae_v1.pt', device='cuda')
anomaly_meta = json.loads((ARTIFACTS / 'anomaly_meta.json').read_text(encoding='utf-8'))
thr = anomaly_meta['convae']['threshold']
print(f'AE loaded, threshold={thr:.4f}')

manual_r = load_retriever(ARTIFACTS / 'rag' / 'manual')
history_r = load_retriever(ARTIFACTS / 'rag' / 'history')
print('retrievers loaded')

configure_tools(ToolDeps(
    sensor_db=db,
    anomaly_model=ae,
    anomaly_threshold=thr,
    anomaly_window=anomaly_meta['window'],
    anomaly_stride=anomaly_meta['stride'],
    manual_retriever=manual_r,
    history_retriever=history_r,
))
print('tools configured')

AE loaded, threshold=0.0742


d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


retrievers loaded
tools configured


d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


## 2. query_sensor

APU-03의 첫 번째 Air Leak 구간(가상 시각: 2020-06-11 ~ 2020-06-13)

In [3]:
out = query_sensor.invoke({
    'equipment_id': 'APU-03',
    'start': '2020-06-11T00:00:00',
    'end': '2020-06-13T00:00:00',
    'sensors': ['TP2', 'Reservoirs', 'Motor_current', 'Oil_temperature'],
})
print(out)

{
  "equipment_id": "APU-03",
  "period": "2020-06-11 00:00:00 ~ 2020-06-12 19:27:00",
  "n_samples": 1770,
  "stats": {
    "TP2": {
      "mean": 4.127922964756524,
      "min": -0.02500000000000035,
      "max": 9.998666666666667,
      "std": 4.502034853795107
    },
    "Reservoirs": {
      "mean": 9.206419556093625,
      "min": 3.841999999999999,
      "max": 10.138,
      "std": 0.585051521100039
    },
    "Motor_current": {
      "mean": 3.3345231705676617,
      "min": 0.034166666666666484,
      "max": 6.25375,
      "std": 2.5835801794644286
    },
    "Oil_temperature": {
      "mean": 63.64215486279257,
      "min": 29.90833333333333,
      "max": 76.05,
      "std": 9.701227234581074
    }
  },
  "fault_label_ratio": null
}


## 3. detect_anomaly

In [4]:
out = detect_anomaly.invoke({
    'equipment_id': 'APU-03',
    'start': '2020-06-11T00:00:00',
    'end': '2020-06-13T00:00:00',
})
print(out)

{
  "equipment_id": "APU-03",
  "period": "2020-06-11 00:00:00 ~ 2020-06-12 19:27:00",
  "n_windows": 58,
  "n_anomaly": 23,
  "anomaly_ratio": 0.39655172413793105,
  "max_score": 0.1965222954750061,
  "threshold": 0.0741691142320633,
  "verdict": "ANOMALY",
  "suspect_sensors": [
    {
      "sensor": "H1",
      "error": 0.29072871804237366
    },
    {
      "sensor": "Motor_current",
      "error": 0.18344703316688538
    },
    {
      "sensor": "TP2",
      "error": 0.17453671991825104
    }
  ]
}


In [ ]:
# 정상 구간
out = detect_anomaly.invoke({
    'equipment_id': 'APU-01',
    'start': '2020-02-15T00:00:00',
    'end': '2020-02-17T00:00:00',
})
print(out)

{
  "equipment_id": "APU-01",
  "period": "2020-02-15 00:00:00 ~ 2020-02-17 00:00:00",
  "n_windows": 95,
  "n_anomaly": 0,
  "anomaly_ratio": 0.0,
  "max_score": 0.013993686065077782,
  "threshold": 0.0741691142320633,
  "verdict": "NORMAL",
  "suspect_sensors": [
    {
      "sensor": "Oil_temperature",
      "error": 0.008821115829050541
    },
    {
      "sensor": "Motor_current",
      "error": 0.008364553563296795
    },
    {
      "sensor": "TP2",
      "error": 0.00535502890124917
    }
  ]
}


## 4. search_manual

In [6]:
out = search_manual.invoke({'query': '공기 누설이 의심될 때 점검 절차', 'k': 2})
print(out)

[
  {
    "id": "SOP-002_air_leak_response#3",
    "source": "SOP-002_air_leak_response.md",
    "title": "SOP-002: 공기 누설(Air Leak) 대응 절차",
    "score": 0.7712,
    "text": "[SOP-002: 공기 누설(Air Leak) 대응 절차 > 3. 2차 점검 (정밀 진단)]\n1. 압축기 정지 후 잔압 0 bar 도달 대기 (안전상 필수)\n2. 초음파 누설 탐지기로 라인 전체 스캔\n3. O-ring / 가스켓 / 솔레노이드 밸브 분해 점검\n4. 토출 밸브(DV) 솔레노이드 동작 시험"
  },
  {
    "id": "SOP-002_air_leak_response#2",
    "source": "SOP-002_air_leak_response.md",
    "title": "SOP-002: 공기 누설(Air Leak) 대응 절차",
    "score": 0.7486,
    "text": "[SOP-002: 공기 누설(Air Leak) 대응 절차 > 2. 1차 점검 (5분 이내 수행)]\n1. 안전 거리에서 청각/시각으로 누설 위치 1차 확인\n2. 토출 밸브 주변 비누 거품 테스트\n3. Reservoirs 라인 플랜지 / 가스켓 육안 점검\n4. 누설 위치 식별되면 단계 4로, 미식별 시 단계 3으로"
  }
]


## 5. search_history

In [7]:
out = search_history.invoke({
    'query': 'TP2 압력 회복 안 됨, 모터 전류 상승',
    'k': 2,
})
print(out)

[
  {
    "case_id": "INC-2020-0701",
    "score": 0.48,
    "metadata": {
      "equipment_id": "APU-02",
      "diagnosis": "공기 누설 + 압축기 부하 증가",
      "date": "2020-07-01T11:45:00",
      "type": "history"
    },
    "text": "[사례 INC-2020-0701] 2020-07-01 11:45 | 설비: APU-02\n증상: Motor_current 피크가 정상 5.2A 대비 7.8A까지 상승, 압력 회복 시간 증가\n진단: 공기 누설 + 압축기 부하 증가\n근본 원인: 흡기 필터 부분 막힘 + DV 솔레노이드 응답 지연 의심\n조치: 흡기 필터 교체, 솔레노이드 코일 측정, 부분 정상화\n다운타임: 50분 / 작업자: 최OO"
  },
  {
    "case_id": "INC-2020-0508",
    "score": 0.4409,
    "metadata": {
      "equipment_id": "APU-02",
      "diagnosis": "공기 누설 (Air Leak)",
      "date": "2020-05-08T03:15:00",
      "type": "history"
    },
    "text": "[사례 INC-2020-0508] 2020-05-08 03:15 | 설비: APU-02\n증상: 야간 무부하 운전 중 TP3 압력 7.2 → 5.8bar 저하 반복\n진단: 공기 누설 (Air Leak)\n근본 원인: Reservoirs 라인 플랜지 가스켓 균열\n조치: 가스켓 신규 교체, 플랜지 토크 재조정, 누설 시험 통과\n다운타임: 120분 / 작업자: 박OO"
  }
]


## 6. draft_workorder

In [8]:
wo = draft_workorder.invoke({
    'equipment_id': 'APU-03',
    'diagnosis': 'Air Leak (DV 토출 밸브 O-ring 노후 추정)',
    'recommended_actions': [
        '잔압 해소 및 LOTO 적용 후 토출 밸브 분해',
        'O-ring 신품 교체 및 시트 청소',
        '재조립 후 30분 무부하 시험 (TP2 ±0.2bar 이내 확인)',
        'CMMS 사례 등록',
    ],
    'priority': 'HIGH',
    'references': ['SOP-002', 'INC-2024-0117'],
})
print(wo)

# 작업지시서 (초안)

| 항목 | 내용 |
|------|------|
| 발급 시각 | 2026-05-05 04:48 |
| 설비 ID | APU-03 |
| 우선순위 | **HIGH** |
| 진단 | Air Leak (DV 토출 밸브 O-ring 노후 추정) |

## 권장 조치
1. 잔압 해소 및 LOTO 적용 후 토출 밸브 분해
2. O-ring 신품 교체 및 시트 청소
3. 재조립 후 30분 무부하 시험 (TP2 ±0.2bar 이내 확인)
4. CMMS 사례 등록

## 참조 문서
- SOP-002
- INC-2024-0117

## 비고
- 본 지시서는 LLM 에이전트가 자동 생성한 초안이며, 작업 전 반드시 담당자 검토 필요.

